In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
df = pd.read_csv("Dataset1.csv")

In [3]:
print("Original Shape:", df.shape)

Original Shape: (186074, 12)


In [4]:
print("Missing Values Per Column:")
print(df.isnull().sum())

Missing Values Per Column:
SN                0
Train_No          0
Station_Code      0
1A                0
2A                0
3A                0
SL                0
Station_Name      0
Route_Number      0
Arrival_time      0
Departure_Time    0
Distance          0
dtype: int64


In [5]:
schedule_cols = ['Arrival_time', 'Departure_Time']

print(df[schedule_cols].isnull().sum())

Arrival_time      0
Departure_Time    0
dtype: int64


In [6]:
df['Arrival_time'] = df['Arrival_time'].replace(
    ['NULL', 'null', '', 'NaN', 'nan'],
    np.nan
)

df['Departure_Time'] = df['Departure_Time'].replace(
    ['NULL', 'null', '', 'NaN', 'nan'],
    np.nan
)

In [7]:
df = df.sort_values(['Train_No', 'SN'])

df['Arrival_time'] = (
    df.groupby('Train_No')['Arrival_time']
      .transform(lambda x: x.ffill().bfill())
)

df['Departure_Time'] = (
    df.groupby('Train_No')['Departure_Time']
      .transform(lambda x: x.ffill().bfill())
)

In [8]:
print("Missing Schedule Values After Cleaning:")
print(df[['Arrival_time','Departure_Time']].isnull().sum())

Missing Schedule Values After Cleaning:
Arrival_time      0
Departure_Time    0
dtype: int64


In [9]:
duplicate_count = df.duplicated().sum()

print("Duplicate Records Found:", duplicate_count)

Duplicate Records Found: 0


In [10]:
df = df.drop_duplicates()

print("Shape After Removing Duplicates:")
print(df.shape)

Shape After Removing Duplicates:
(186074, 12)


In [11]:
print("Remaining Duplicates:",
      df.duplicated().sum())

Remaining Duplicates: 0


In [12]:
def check_sn_order(group):
    return group['SN'].is_monotonic_increasing

sn_validation = (
    df.groupby('Train_No')
      .apply(check_sn_order)
      .reset_index(name='SN_Order_Correct')
)

print(sn_validation.head())

   Train_No  SN_Order_Correct
0       107              True
1       108              True
2       128              True
3       290              True
4       401              True


/var/folders/0c/7f8p6mvs5mg8nbh9mllqtkb80000gn/T/ipykernel_1776/3938436534.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(check_sn_order)


In [13]:
def check_distance_order(group):
    return group['Distance'].is_monotonic_increasing

distance_validation = (
    df.groupby('Train_No')
      .apply(check_distance_order)
      .reset_index(name='Distance_Order_Correct')
)

print(distance_validation.head())

   Train_No  Distance_Order_Correct
0       107                    True
1       108                    True
2       128                    True
3       290                    True
4       401                    True


/var/folders/0c/7f8p6mvs5mg8nbh9mllqtkb80000gn/T/ipykernel_1776/2582996317.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(check_distance_order)


In [14]:
validation_report = pd.merge(
    sn_validation,
    distance_validation,
    on='Train_No'
)

validation_report['Route_Valid'] = (
    validation_report['SN_Order_Correct']
    &
    validation_report['Distance_Order_Correct']
)

print(validation_report.head())

   Train_No  SN_Order_Correct  Distance_Order_Correct  Route_Valid
0       107              True                    True         True
1       108              True                    True         True
2       128              True                    True         True
3       290              True                    True         True
4       401              True                    True         True


In [15]:
invalid_routes = validation_report[
    validation_report['Route_Valid'] == False
]

print("Invalid Routes Found:")
print(invalid_routes)

Invalid Routes Found:
Empty DataFrame
Columns: [Train_No, SN_Order_Correct, Distance_Order_Correct, Route_Valid]
Index: []


In [16]:
valid_train_numbers = validation_report[
    validation_report['Route_Valid']
]['Train_No']

verified_df = df[
    df['Train_No'].isin(valid_train_numbers)
]

print("Verified Dataset Shape:")
print(verified_df.shape)

Verified Dataset Shape:
(186074, 12)


In [17]:
verified_df.to_csv(
    "Verified_Train_Dataset.csv",
    index=False
)

print("Verified dataset saved successfully.")

Verified dataset saved successfully.


In [18]:
print("\n===== DATA QUALITY REPORT =====")

print("Original Records :", len(df))

print("Verified Records :", len(verified_df))

print("Valid Routes :",
      validation_report['Route_Valid'].sum())

print("Invalid Routes :",
      (~validation_report['Route_Valid']).sum())

print("Remaining Missing Values :")
print(
    verified_df[
        ['Arrival_time','Departure_Time']
    ].isnull().sum()
)


===== DATA QUALITY REPORT =====
Original Records : 186074
Verified Records : 186074
Valid Routes : 11113
Invalid Routes : 0
Remaining Missing Values :
Arrival_time      0
Departure_Time    0
dtype: int64
